# 05 · Milestone 2: baseline generation and CFG / scheduler / steps sweeps

This notebook runs 100 seeded generations (seed 42) over the 5 Milestone 1 prompts:

| Sweep | Settings | Images |
|---|---|---|
| CLIP vs. projected BERT | PNDM, 30 steps, s = 7.5 | 10 |
| Guidance scale | s ∈ {1, 3, 5, 7.5, 10, 15} | 30 |
| Noise scheduler | PNDM / DDIM / DPM-Solver++ / Euler | 20 |
| Sampling steps | {10, 20, 30, 50} for DDIM and DPM | 40 |

Each image is then scored with **CLIP score** (ViT-B/32). Run notebook 04 first. Runtime is about 15–20 min on a T4. If Colab disconnects, Run all again: finished images are reused.

In [ ]:
# --- Setup: mount Drive, clone or update the repo, install pinned deps. Safe to rerun. ---
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Pranav1011/ie7615-generative-project.git"
BRANCH = "main"
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

def _git(*args):
    """Run git. For a private repo, add a GITHUB_TOKEN Colab secret (key icon in the sidebar).

    The token goes in a one-off HTTP header, so it is never written to .git/config."""
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    auth = []
    if token:
        import base64
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        auth = ["-c", f"http.https://github.com/.extraheader=Authorization: Basic {basic}"]
    res = subprocess.run(["git", *auth, *args], capture_output=True, text=True)
    out = res.stdout + res.stderr
    if token:
        out = out.replace(token, "***").replace(basic, "***")
    if res.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/ie7615-generative-project")
    if (REPO_DIR / ".git").exists():
        print(_git("-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH))
    else:
        _git("clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("repo:", REPO_DIR, "| commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
from src.utils import load_config, get_paths, set_seed
cfg = load_config()
paths = get_paths(cfg)
set_seed(cfg["seed"])
print("artifact root:", paths["root"])

In [ ]:
assert paths["bert_adapter_ckpt"].exists(), "Run notebook 04 first (adapter checkpoint missing)."
from src.generate import load_pipeline
from src.sweeps import run_milestone2_sweeps

pipe = load_pipeline(cfg)
rows = run_milestone2_sweeps(pipe, cfg, paths)

## CLIP score

In [ ]:
import gc, torch
import pandas as pd
from src.metrics import ClipScorer, score_runs, summarize_runs
from src.sweeps import LOG_FIELDS, write_csv

del pipe
gc.collect(); torch.cuda.empty_cache()
sw = cfg["milestone2_sweeps"]
scorer = ClipScorer(sw["eval_clip_model"], revision=sw["eval_clip_revision"])
rows = score_runs(rows, paths["milestone2_samples"], scorer)
logs = paths["milestone2_logs"]
write_csv(rows, logs / "sweep_runs_log.csv", LOG_FIELDS + ["clip_score", "clip_score_model"])
summary = summarize_runs(rows)
summary.to_csv(logs / "sweep_summary.csv", index=False)
display(summary)

In [ ]:
import matplotlib.pyplot as plt

df = pd.DataFrame(rows)
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

cfg_df = df[df.sweep == "cfg_scale"]
for pid, g in cfg_df.groupby("prompt_id"):
    axes[0, 0].plot(g.guidance_scale, g.clip_score, marker="o", alpha=0.45, label=pid)
axes[0, 0].plot(*zip(*cfg_df.groupby("guidance_scale").clip_score.mean().items()), "k-o", lw=2.5, label="mean")
axes[0, 0].set(title="CLIP score vs guidance scale (PNDM, 30 steps)", xlabel="guidance scale s", ylabel="CLIP score")

st_df = df[df.sweep == "steps"]
for sch, g in st_df.groupby("scheduler"):
    mean = g.groupby("steps").clip_score.mean()
    axes[0, 1].plot(mean.index, mean.values, marker="o", label=sch)
axes[0, 1].set(title="CLIP score vs sampling steps (s = 7.5, mean of 5 prompts)", xlabel="steps")

sc_df = df[df.sweep == "scheduler"].groupby("scheduler").agg(score=("clip_score", "mean"), sec=("elapsed_sec", "mean"))
axes[1, 0].bar(sc_df.index, sc_df["score"], color="#3b6ea5")
for i, (c, s) in enumerate(zip(sc_df["score"], sc_df["sec"])):
    axes[1, 0].text(i, c, f"{c:.1f}\n{s:.1f}s", ha="center", va="bottom", fontsize=9)
axes[1, 0].set(title="Scheduler (30 steps, s = 7.5): mean CLIP score and time", ylim=(0, df.clip_score.max() * 1.2))

bl = df[df.sweep == "baseline"].pivot(index="prompt_id", columns="encoder", values="clip_score")
bl.plot.bar(ax=axes[1, 1], color=["#c0504d", "#3b6ea5"], rot=20)
axes[1, 1].set(title="CLIP vs projected-BERT conditioning, per prompt", xlabel="")
for ax in axes.flat:
    ax.grid(alpha=0.3)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(logs / "clip_score_plots.png", dpi=150)
plt.show()

## Sample grids

In [ ]:
from IPython.display import display as show
from PIL import Image
for name in ("grid_clip_vs_bert.jpg", "grid_cfg_sweep.jpg", "grid_scheduler_sweep.jpg", "grid_steps_sweep.jpg"):
    print(name)
    show(Image.open(paths["milestone2_samples"] / name))

## Download the commit-worthy results
The zip holds the logs, plots, grids and the 10 baseline images (CLIP and BERT). It leaves out the other 90 sweep images, which stay on Drive. Send the zip to whoever commits the results.

In [ ]:
import shutil, zipfile
from pathlib import Path

samples, logs = paths["milestone2_samples"], paths["milestone2_logs"]
keep = [p for p in logs.iterdir() if p.suffix in (".csv", ".json", ".png")]
keep += sorted(samples.glob("grid_*.jpg")) + sorted((samples / "baseline").rglob("*.png")) + sorted((samples / "baseline").rglob("*.json"))
zip_path = Path("/content/milestone2_results.zip" if IN_COLAB else paths["outputs"] / "milestone2_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in keep:
        zf.write(p, ("logs/" if p.parent == logs else "samples/") + str(p.relative_to(logs if p.parent == logs else samples)))
print(f"{len(keep)} files -> {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))